# Breast Cancer Detection — Notebook 2
## EfficientNet-B0 + Focal Loss on BUSI (class-imbalance handling)

Same BUSI pipeline as Notebook 1, but with **EfficientNet-B0** and a **Focal Loss** (with positive-class weighting computed from the training distribution) to deal with the benign/malignant imbalance. Includes a decision-threshold analysis, which matters in a medical setting where recall on the malignant class is the priority.

### 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive/breast_cancer_project"

for folder in ["data", "models", "results", "plots"]:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Project folders ready")

In [ ]:
from google.colab import files
files.upload()   # upload your kaggle.json here

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

### 2. Dataset

In [ ]:
!kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset
!unzip -q breast-ultrasound-images-dataset.zip -d data

In [ ]:
import pandas as pd
from glob import glob

paths = glob("data/Dataset_BUSI_with_GT/*/*.png")

# keep only the original scans, drop the ground-truth segmentation masks
paths = [p for p in paths if "_mask" not in p]

df = pd.DataFrame({"path": paths})
df["class"] = df["path"].apply(lambda p: p.split("/")[-2])

# Binary target: malignant = 1, benign / normal = 0
df["label"] = (df["class"] == "malignant").astype(int)

print(df["class"].value_counts())
print()
print(df["label"].value_counts())
df.head()

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.30,
                                     stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50,
                                   stratify=temp_df["label"], random_state=42)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Malignant ratio (train):", round(train_df["label"].mean(), 3))

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class BreastUltrasoundDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = row["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(BreastUltrasoundDataset(train_df, train_transform),
                          batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(BreastUltrasoundDataset(val_df, val_transform),
                        batch_size=16, num_workers=2)
test_loader = DataLoader(BreastUltrasoundDataset(test_df, val_transform),
                         batch_size=16, num_workers=2)

### 3. Model — EfficientNet-B0 + Focal Loss

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.efficientnet_b0(pretrained=True)

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 1)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
import torch.nn as nn

class FocalLoss(nn.Module):
    """Binary focal loss with optional positive-class weighting."""

    def __init__(self, alpha=0.75, gamma=2.0, pos_weight=None):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        if pos_weight is not None:
            pos_weight = torch.tensor([pos_weight], device=device)
        self.bce = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets.float())
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

In [ ]:
# positive-class weight from the training distribution
n_pos = int(train_df["label"].sum())
n_neg = len(train_df) - n_pos
pos_weight = n_neg / n_pos
print(f"pos: {n_pos}, neg: {n_neg}, pos_weight: {pos_weight:.2f}")

criterion = FocalLoss(alpha=0.75, gamma=2.0, pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

### 4. Training

In [ ]:
EPOCHS = 15
history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.float().to(device)

        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Epoch {epoch+1}, Step {i}, Loss: {loss.item():.4f}")

    # ---------------- validation ----------------
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            val_loss += criterion(outputs, labels).item()

            preds = (torch.sigmoid(outputs) > 0.5).int()
            correct += (preds == labels.int()).sum().item()
            total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), base_path + "/models/efficientnet_focal_busi_best.pth")
        print("  -> new best model saved")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], color="#0072B2", lw=2, label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], color="#D55E00", lw=2, label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()

axes[1].plot(epochs_range, history["val_acc"], color="#009E73", lw=2, label="Val accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/efficientnet_focal_busi_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5. Evaluation + threshold analysis

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, accuracy_score)

model.load_state_dict(torch.load(base_path + "/models/efficientnet_focal_busi_best.pth"))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(np.array(labels))

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(" thr | precision | recall |   f1")
print("-----+-----------+--------+------")
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (all_probs > thr).astype(int)
    print(f" {thr:.1f} |   {precision_score(all_labels, p, zero_division=0):.3f}   "
          f"| {recall_score(all_labels, p):.3f}  | {f1_score(all_labels, p):.3f}")

In [ ]:
# chosen operating point: favour recall on the malignant class
THRESHOLD = 0.4

preds = (all_probs > THRESHOLD).astype(int)

acc = accuracy_score(all_labels, preds)
auc = roc_auc_score(all_labels, all_probs)

print("Threshold:", THRESHOLD)
print("Test Accuracy:", round(acc, 4))
print("ROC-AUC:", round(auc, 4))
print("\nConfusion Matrix:\n", confusion_matrix(all_labels, preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, preds,
                            target_names=["benign/normal", "malignant"]))

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion matrix (single-hue sequential colormap)
cm = confusion_matrix(all_labels, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["benign/normal", "malignant"],
            yticklabels=["benign/normal", "malignant"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="#0072B2", lw=2, label=f"ROC (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Chance")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/efficientnet_focal_busi_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Save artifacts to Drive

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
f1 = f1_score(all_labels, preds)

results_text = f"""MODEL: EfficientNet-B0 + Focal Loss
DATASET: BUSI — Breast Ultrasound Images (binary)
LOSS: Focal Loss (alpha=0.75, gamma=2.0, pos_weight from data)
THRESHOLD: {THRESHOLD}

Test Accuracy: {acc:.4f}
ROC-AUC: {auc:.4f}

Malignant class:
Precision: {precision:.4f}
Recall: {recall:.4f}
F1-score: {f1:.4f}

Confusion Matrix:
{confusion_matrix(all_labels, preds)}

Classification Report:
{classification_report(all_labels, preds)}
"""

with open(base_path + "/results/efficientnet_focal_busi_results.txt", "w") as f:
    f.write(results_text)

print(results_text)
print("Results saved")

In [ ]:
threshold_text = """THRESHOLD ANALYSIS (EfficientNet-B0 + Focal Loss, BUSI):

0.2 -> highest recall, lowest precision
0.3 -> high recall, precision still low
0.4 -> selected threshold (recall-first, acceptable precision)
0.5 -> balanced default
0.6 -> conservative, recall drops
0.7 -> too conservative for a screening use case

FINAL CHOICE: 0.4 (missing a malignant case is costlier
than a false alarm in a screening context)
"""

with open(base_path + "/results/efficientnet_focal_busi_threshold_analysis.txt", "w") as f:
    f.write(threshold_text)

print("Threshold analysis saved")

In [ ]:
results_df = pd.DataFrame({
    "path": test_df["path"].values,
    "true_label": all_labels,
    "predicted_label": preds,
    "probability": all_probs
})

results_df.to_csv(base_path + "/results/efficientnet_focal_busi_predictions.csv", index=False)
print("Predictions saved")

In [ ]:
import json

config = {
    "model": "EfficientNet-B0",
    "pretrained": "ImageNet",
    "dataset": "BUSI (aryashah2k/breast-ultrasound-images-dataset)",
    "task": "binary classification (malignant vs benign/normal)",
    "loss": "Focal Loss",
    "alpha": 0.75,
    "gamma": 2.0,
    "pos_weight": "n_neg / n_pos (computed on train split)",
    "optimizer": "Adam",
    "lr": 1e-4,
    "epochs": 15,
    "batch_size": 16,
    "image_size": 224,
    "threshold": 0.4,
    "seed": 42
}

with open(base_path + "/results/efficientnet_focal_busi_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Config saved")